In [8]:
import torch
import transformers
print("Torch version:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

Torch version: 2.6.0+cpu
CUDA disponível: False


[I 2025-04-01 08:21:56,634] A new study created in memory with name: no-name-883fb579-5b28-43d8-b41e-02dba134cba9
[I 2025-04-01 08:21:56,655] Trial 0 finished with value: 19.849899330880017 and parameters: {'x': 6.4553225843792745}. Best is trial 0 with value: 19.849899330880017.
[I 2025-04-01 08:21:56,657] Trial 1 finished with value: 95.91101973765316 and parameters: {'x': -7.7934171634651195}. Best is trial 0 with value: 19.849899330880017.
[I 2025-04-01 08:21:56,659] Trial 2 finished with value: 25.683749121069656 and parameters: {'x': -3.067913685242642}. Best is trial 0 with value: 19.849899330880017.
[I 2025-04-01 08:21:56,660] Trial 3 finished with value: 4.991335540841943 and parameters: {'x': -0.23412970546518252}. Best is trial 3 with value: 4.991335540841943.
[I 2025-04-01 08:21:56,662] Trial 4 finished with value: 130.08896422448277 and parameters: {'x': -9.405654923084548}. Best is trial 3 with value: 4.991335540841943.
[I 2025-04-01 08:21:56,664] Trial 5 finished with va

{'x': 2.0038626308074297}

In [3]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import random

# Caminhos para os dados
DATASET_PATH = r"C:\Users\paulo\Downloads\cell_images\cell_images"
INFECTED_PATH = os.path.join(DATASET_PATH, "Parasitized")  # Células infectadas
UNINFECTED_PATH = os.path.join(DATASET_PATH, "Uninfected")  # Células saudáveis

# Transformações de imagem para Swin Transformer
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Tamanho esperado pelo Swin Transformer
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Criar dataset personalizado
class MalariaDataset(Dataset):
    def __init__(self, data_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        self.images = []
        self.labels = []
        
        # Carregar imagens infectadas (1)
        for img_name in os.listdir(INFECTED_PATH):
            self.images.append(os.path.join(INFECTED_PATH, img_name))
            self.labels.append(1)
        
        # Carregar imagens não infectadas (0)
        for img_name in os.listdir(UNINFECTED_PATH):
            self.images.append(os.path.join(UNINFECTED_PATH, img_name))
            self.labels.append(0)

        # Embaralhar os dados
        temp = list(zip(self.images, self.labels))
        random.shuffle(temp)
        self.images, self.labels = zip(*temp)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

# Criar instâncias do dataset
dataset = MalariaDataset(DATASET_PATH, transform=transform)

# Dividir em treino e teste (80% treino, 20% teste)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Tamanho do dataset: {len(dataset)}")
print(f"Imagens de treino: {len(train_dataset)}, Imagens de teste: {len(test_dataset)}")


Tamanho do dataset: 27560
Imagens de treino: 22048, Imagens de teste: 5512


In [12]:
from transformers import SwinForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
import os
from PIL import Image

DATASET_PATH = r"C:\Users\paulo\Downloads\cell_images\cell_images"

# Listar arquivos que podem ser problemáticos
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        try:
            with Image.open(file_path) as img:
                img.verify()  # Verifica se a imagem é válida
        except Exception as e:
            print(f"Erro no arquivo: {file_path} - {str(e)}")


Erro no arquivo: C:\Users\paulo\Downloads\cell_images\cell_images\Parasitized\Thumbs.db - cannot identify image file 'C:\\Users\\paulo\\Downloads\\cell_images\\cell_images\\Parasitized\\Thumbs.db'
Erro no arquivo: C:\Users\paulo\Downloads\cell_images\cell_images\Uninfected\Thumbs.db - cannot identify image file 'C:\\Users\\paulo\\Downloads\\cell_images\\cell_images\\Uninfected\\Thumbs.db'


In [17]:
for folder in ["Parasitized", "Uninfected"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    for file in os.listdir(folder_path):
        ext = os.path.splitext(file)[-1].lower()
        if ext not in [".png", ".jpg", ".jpeg"]:
            print(f"Arquivo inválido detectado: {file}")

Arquivo inválido detectado: Thumbs.db
Arquivo inválido detectado: Thumbs.db


### Treinando o modelo

In [ ]:

import optuna
from sklearn.metrics import f1_score
import torch.nn as nn
import torch.optim as optim

# Função de treino simplificada
def train_one_epoch(model, optimizer, criterion, loader, trial):
    model.train()
    loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        # trial.report(loss, "mudar este label loss")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss

# Função de avaliação: retorna F1-score
def evaluate(model, loader, trial):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    score = f1_score(all_labels, all_preds)
    # trial.report(score, "mudar este label")
        # Handle pruning based on the intermediate value.
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()
    return score

def evaluate_metrics(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    return accuracy, precision, recall, f1, tp, fp, tn, fn

# Função objetivo para o Optuna
def objective(trial):
    # Sugerir hiperparâmetros
    lr = trial.suggest_float("lr", 1e-6, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    # otimizadores: AdamW, SGD, Adam
    optimizer_type = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])
    loss_fn = trial.suggest_categorical("loss_fn", ["CrossEntropy", "BCE"])
    
    
    # Estrutura do modelo base
    model = SwinForImageClassification.from_pretrained("microsoft/swin-tiny-patch4-window7-224", num_labels=2, ignore_mismatched_sizes=True)

    # Carregar pesos salvos localmente
    model_path= r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\swin-variations\swin-sem-optuna\modelos_salvos_swin\swin_fold1.pth"
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)

    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])  # for hp tuning
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    # Treinar por poucas épocas (ajustar conforme o tempo disponível)
    for epoch in range(3):  # pode aumentar se quiser mais estabilidade
        #monitora a loss
        loss = train_one_epoch(model, optimizer, criterion, train_loader, trial)
    
    # Avaliar
    f1 = evaluate(model, test_loader, trial)
    return f1



study = optuna.create_study(direction="maximize")  # 'maximize' because objective function is returning accuracy
# study = optuna.create_study(direction="minimize")  # 'minimize' because objective function is returning loss
study.optimize(objective, n_trials=30)

pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))


[I 2025-04-01 11:50:55,562] A new study created in memory with name: no-name-f07de3ce-97b0-4d68-bc71-b77926964c0b
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:

# Treinar o modelo final com os melhores hiperparâmetros do Optuna
best_params = study.best_params

# Recarregar modelo limpo
final_model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-tiny-patch4-window7-224", 
    num_labels=2, 
    ignore_mismatched_sizes=True
).to(device)

# Recriar otimizador com os melhores parâmetros
final_optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"]
)

final_criterion = nn.CrossEntropyLoss()

# Treinar modelo final por mais épocas
print("Treinando modelo final com os melhores hiperparâmetros:")
for epoch in range(10):  # agora podemos treinar mais profundamente
    train_one_epoch(final_model, final_optimizer, final_criterion, train_loader)
    f1 = evaluate(final_model, test_loader)
    print(f"Época {epoch+1}: F1-score = {f1:.4f}")

# Você pode salvar o modelo com:
# torch.save(final_model.state_dict(), "melhor_modelo_swin.pth")


Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Treinando modelo final com os melhores hiperparâmetros:
Época 1: F1-score = 0.9647
Época 2: F1-score = 0.9719
Época 3: F1-score = 0.9700
Época 4: F1-score = 0.9726
Época 5: F1-score = 0.9687
Época 6: F1-score = 0.9699
Época 7: F1-score = 0.9681
Época 8: F1-score = 0.9702
Época 9: F1-score = 0.9719
Época 10: F1-score = 0.9711


In [10]:

import pandas as pd
from sklearn.metrics import precision_score, recall_score, accuracy_score, confusion_matrix
import os

# Lista para armazenar os resultados dos trials
optuna_results_list = []

# Atualização da função de avaliação com métricas completas
def evaluate_metrics(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).logits
            preds = torch.argmax(outputs, dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    return accuracy, precision, recall, f1, tp, fp, tn, fn

# Redefinir a função objetivo com logging
def objective(trial):
    lr = trial.suggest_float("lr", 1e-6, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    
    model = SwinForImageClassification.from_pretrained(
        "microsoft/swin-tiny-patch4-window7-224",
        num_labels=2,
        ignore_mismatched_sizes=True
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    model_name = "swin-tiny"
    fold = trial.number

    for epoch in range(3):  # ou mais se quiser
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).logits
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Avaliar após cada época
        accuracy, precision, recall, f1, tp, fp, tn, fn = evaluate_metrics(model, test_loader)

        # Log dos resultados por época
        optuna_results_list.append({
            "fold": fold,
            "epoch": epoch + 1,
            "model": model_name,
            "loss": running_loss / len(train_loader),
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "true_positives": tp,
            "false_positives": fp,
            "true_negatives": tn,
            "false_negatives": fn,
            "learning_rate": lr
        })

    return f1

# Rodar o estudo
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

# Salvar métricas do Optuna em CSV
optuna_df = pd.DataFrame(optuna_results_list)
os.makedirs("logs", exist_ok=True)
optuna_df.to_csv("logs/optuna_trials_results.csv", index=False)
print("Resultados dos trials salvos em logs/optuna_trials_results.csv")


[I 2025-03-31 09:44:14,618] A new study created in memory with name: no-name-d750f72a-017e-450b-b0da-fcf66eca17c3
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-03-31 09:54:48,278] Trial 0 finished with value: 0.6611610395919358 and parameters: {'lr': 0.0005061152562846902, 'weight_decay': 5.129884194254795e-05}. Best is trial 0 with value: 0.6611610395919358.
Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-tiny-patch4-window7-224 a

Resultados dos trials salvos em logs/optuna_trials_results.csv
